## DATA 1.2 | The power of Feature Engineering

This notebook was made for a lecture on feature engineering at the University of São Paulo for the members of the Data Science and Machine Learning group. It is written in Portuguese.

## Importando bibliotecas

In [1]:
!pip install catboost==0.15.2

     |████████████████████████████████| 61.2MB 51.2MB/s 
  Found existing installation: catboost 0.18
    Uninstalling catboost-0.18:
      Successfully uninstalled catboost-0.18


In [2]:
import numpy as np
import pandas as pd
import os
import gc

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
import xgboost as xgb
import catboost as cb

## Funções úteis

In [3]:
def reduce_mem_usage(df):
    """ iterate through all the columns of a dataframe and modify the data type
        to reduce memory usage.        
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df

## Feature selection (tema não abordado na aula)

In [4]:
# COLUMNS WITH STRINGS
str_type = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain','M1', 'M2', 'M3', 'M4','M5',
            'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 
            'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']

# FIRST 53 COLUMNS
cols = ['TransactionID', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
       'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain',
       'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11',
       'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8',
       'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4',
       'M5', 'M6', 'M7', 'M8', 'M9']

# V COLUMNS TO LOAD DECIDED BY CORRELATION EDA
# https://www.kaggle.com/cdeotte/eda-for-columns-v-and-id
v =  [1, 3, 4, 6, 8, 11]
v += [13, 14, 17, 20, 23, 26, 27, 30]
v += [36, 37, 40, 41, 44, 47, 48]
v += [54, 56, 59, 62, 65, 67, 68, 70]
v += [76, 78, 80, 82, 86, 88, 89, 91]

#v += [96, 98, 99, 104] #relates to groups, no NAN 
v += [107, 108, 111, 115, 117, 120, 121, 123] # maybe group, no NAN
v += [124, 127, 129, 130, 136] # relates to groups, no NAN

# LOTS OF NAN BELOW
v += [138, 139, 142, 147, 156, 162] #b1
v += [165, 160, 166] #b1
v += [178, 176, 173, 182] #b2
v += [187, 203, 205, 207, 215] #b2
v += [169, 171, 175, 180, 185, 188, 198, 210, 209] #b2
v += [218, 223, 224, 226, 228, 229, 235] #b3
v += [240, 258, 257, 253, 252, 260, 261] #b3
v += [264, 266, 267, 274, 277] #b3
v += [220, 221, 234, 238, 250, 271] #b3

v += [294, 284, 285, 286, 291, 297] # relates to grous, no NAN
v += [303, 305, 307, 309, 310, 320] # relates to groups, no NAN
v += [281, 283, 289, 296, 301, 314] # relates to groups, no NAN
#v += [332, 325, 335, 338] # b4 lots NAN

cols += ['V'+str(x) for x in v]
dtypes = {}
for c in cols+['id_0'+str(x) for x in range(1,10)]+['id_'+str(x) for x in range(10,34)]: 
    dtypes[c] = 'float32'
for c in str_type: dtypes[c] = 'object'

## Carregando os dados

In [5]:
%%time

folder_path = '../input/ieee-fraud-detection/'
print('Loading data...')

train_identity = pd.read_csv(f'{folder_path}train_identity.csv', index_col='TransactionID',  dtype=dtypes)
print('\tSuccessfully loaded train_identity!')

train_transaction = pd.read_csv(f'{folder_path}train_transaction.csv', index_col='TransactionID', dtype=dtypes, usecols=cols+['isFraud'])
print('\tSuccessfully loaded train_transaction!')

test_identity = pd.read_csv(f'{folder_path}test_identity.csv', index_col='TransactionID', dtype=dtypes)
print('\tSuccessfully loaded test_identity!')

test_transaction = pd.read_csv(f'{folder_path}test_transaction.csv', index_col='TransactionID', dtype=dtypes, usecols=cols)
print('\tSuccessfully loaded test_transaction!')

sub = pd.read_csv(f'{folder_path}sample_submission.csv')
print('\tSuccessfully loaded sample_submission!')

print('Data was successfully loaded!\n')

Loading data...
	Successfully loaded train_identity!
	Successfully loaded train_transaction!
	Successfully loaded test_identity!
	Successfully loaded test_transaction!
	Successfully loaded sample_submission!
Data was successfully loaded!

CPU times: user 28.6 s, sys: 1.91 s, total: 30.5 s
Wall time: 30.8 s


## Juntando os dados

In [6]:
print('Merging data...')
train = train_transaction.merge(train_identity, how='left', left_index=True, right_index=True)
test = test_transaction.merge(test_identity, how='left', left_index=True, right_index=True)

print('Data was successfully merged!\n')

del train_identity, train_transaction, test_identity, test_transaction

print(f'Train dataset has {train.shape[0]} rows and {train.shape[1]} columns.')
print(f'Test dataset has {test.shape[0]} rows and {test.shape[1]} columns.')

gc.collect()

Merging data...
Data was successfully merged!

Train dataset has 590540 rows and 214 columns.
Test dataset has 506691 rows and 213 columns.


0

In [7]:
train.head()

,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
TransactionID,,,,,,,,,,,,,,,,,,,,,
2987000.0,0,86400.0,68.5,W,13926.0,NaN,150.0,discover,142.0,credit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2987001.0,0,86401.0,29.0,W,2755.0,404.0,150.0,mastercard,102.0,credit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2987002.0,0,86469.0,59.0,W,4663.0,490.0,150.0,visa,166.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2987003.0,0,86499.0,50.0,W,18132.0,567.0,150.0,mastercard,117.0,debit,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2987004.0,0,86506.0,50.0,H,4497.0,514.0,150.0,mastercard,102.0,credit,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [8]:
test.head()

,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
TransactionID,,,,,,,,,,,,,,,,,,,,,
3663549.0,18403224.0,31.950001,W,10409.0,111.0,150.0,visa,226.0,debit,170.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3663550.0,18403264.0,49.000000,W,4272.0,111.0,150.0,visa,226.0,debit,299.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3663551.0,18403310.0,171.000000,W,4476.0,574.0,150.0,visa,226.0,debit,472.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3663552.0,18403310.0,284.950012,W,10989.0,360.0,150.0,visa,166.0,debit,205.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3663553.0,18403316.0,67.949997,W,18018.0,452.0,150.0,mastercard,117.0,debit,264.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Feature Engineering

### Normalizando variáveis dependentes de tempo

In [9]:
for i in range(1,16):
    if i in [1,2,3,5,9]: continue
    train['D'+str(i)] =  train['D'+str(i)] - train.TransactionDT/np.float32(24*60*60)
    test['D'+str(i)] = test['D'+str(i)] - test.TransactionDT/np.float32(24*60*60)

### Frequency encoding

In [10]:
# Demonstrando o que value_counts retorna
pd.concat([train['card1'], test['card1']], ignore_index=True).value_counts(dropna=False)

7919.0     28015
9500.0     26243
15885.0    22691
17188.0    19606
15066.0    14606
           ...  
16323.0        1
16324.0        1
16331.0        1
16337.0        1
8192.0         1
Name: card1, Length: 17091, dtype: int64

In [11]:
# Realizando encoding
for feature in ['card1', 'card2', 'card3', 'P_emaildomain']:
    train[feature + '_count'] = train[feature].map(pd.concat([train[feature], test[feature]], ignore_index=True).value_counts(dropna=False))
    test[feature + '_count'] = test[feature].map(pd.concat([train[feature], test[feature]], ignore_index=True).value_counts(dropna=False))

Pode causar data leakage, vale tentar implementar de várias formas.

### Aggregations

In [12]:
# Apenas para demonstrar o que o groupby faz, não vamos usar como feature agora.
train.groupby('card1')['TransactionAmt'].agg(['mean'])

,mean
card1,
1000.0,23.443001
1001.0,79.666664
1004.0,136.399994
1005.0,50.000000
1006.0,133.333328
...,...
18390.0,56.806408
18391.0,171.000000
18393.0,321.816681


### Criando dia e hora da transação

In [13]:
def make_day_feature(df, offset=0, tname='TransactionDT'):
    days = df[tname] / (3600*24)        
    encoded_days = np.floor(days-1+offset) % 7
    return encoded_days

def make_hour_feature(df, tname='TransactionDT'):
    hours = df[tname] / (3600)        
    encoded_hours = np.floor(hours) % 24
    return encoded_hours

In [14]:
train['weekday'] = make_day_feature(train, offset=0.58)
test['weekday'] = make_day_feature(test, offset=0.58)

train['hours'] = make_hour_feature(train)
test['hours'] = make_hour_feature(test)

## Encoding

In [15]:
%%time

cat_features = []

for col in train.columns:
    if train[col].dtype == 'object':
        le = LabelEncoder()
        le.fit(list(train[col].astype(str).values) + list(test[col].astype(str).values))
        train[col] = le.transform(list(train[col].astype(str).values))
        test[col] = le.transform(list(test[col].astype(str).values))
        cat_features.append(col)

CPU times: user 1min 1s, sys: 3.97 s, total: 1min 5s
Wall time: 1min 5s


In [16]:
import datetime
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')

train['DT_M'] = train['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
train['DT_M'] = (train['DT_M'].dt.year-2017)*12 + train['DT_M'].dt.month 

test['DT_M'] = test['TransactionDT'].apply(lambda x: (START_DATE + datetime.timedelta(seconds = x)))
test['DT_M'] = (test['DT_M'].dt.year-2017)*12 + test['DT_M'].dt.month

## Preparando dados para o modelo

In [17]:
train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

Memory usage of dataframe is 607.96 MB
Memory usage after optimization is: 275.69 MB
Decreased by 54.7%
Memory usage of dataframe is 520.61 MB
Memory usage after optimization is: 241.80 MB
Decreased by 53.6%


In [18]:
X = train.sort_values('TransactionDT').drop(['isFraud', 'TransactionDT'], axis=1)
y = train.sort_values('TransactionDT')['isFraud']

X_test = test.drop(['TransactionDT'], axis=1)

del train, test
gc.collect()

0

In [19]:
X = X.fillna(-1)
X_test = X_test.fillna(-1)

cols = X.columns

## XGBoost

In [20]:
xgb_params = {'n_estimators':5000,
              'max_depth':12,
              'learning_rate':0.02,
              'subsample':0.8,
              'colsample_bytree':0.4,
              'missing':-1,
              'eval_metric':'auc',
              'tree_method':'gpu_hist'
             }

In [21]:
%%time

oof = np.zeros(len(X))
preds = np.zeros(len(X_test))

skf = GroupKFold(n_splits=6)
for i, (idxT, idxV) in enumerate(skf.split(X, y, groups=X['DT_M'])):
    month = X.iloc[idxV]['DT_M'].iloc[0]
    print('Fold',i,'withholding month',month)
    print('rows of train =',len(idxT),'rows of holdout =',len(idxV))
    
    clf = xgb.XGBClassifier(**xgb_params)

    h = clf.fit(X[cols].iloc[idxT], y.iloc[idxT], 
            eval_set=[(X[cols].iloc[idxV],y.iloc[idxV])],
            verbose=100, early_stopping_rounds=200)

    oof[idxV] += clf.predict_proba(X[cols].iloc[idxV])[:,1]
    preds += clf.predict_proba(X_test[cols])[:,1]/skf.n_splits
    
    del h, clf
    x = gc.collect()

print('#'*20)
print ('XGB OOF CV=',roc_auc_score(y,oof))

Fold 0 withholding month 12
rows of train = 453219 rows of holdout = 137321
[0]	validation_0-auc:0.816732
Will train until validation_0-auc hasn't improved in 200 rounds.
[100]	validation_0-auc:0.866072
[200]	validation_0-auc:0.883784
[300]	validation_0-auc:0.897484
[400]	validation_0-auc:0.904751
[500]	validation_0-auc:0.907243
[600]	validation_0-auc:0.908056
[700]	validation_0-auc:0.908415
[800]	validation_0-auc:0.908769
[900]	validation_0-auc:0.908062
Stopping. Best iteration:
[779]	validation_0-auc:0.908866

Fold 1 withholding month 15
rows of train = 488908 rows of holdout = 101632
[0]	validation_0-auc:0.830833
Will train until validation_0-auc hasn't improved in 200 rounds.
[100]	validation_0-auc:0.897488
[200]	validation_0-auc:0.919382
[300]	validation_0-auc:0.932742
[400]	validation_0-auc:0.939009
[500]	validation_0-auc:0.941399
[600]	validation_0-auc:0.941979
[700]	validation_0-auc:0.942554
[800]	validation_0-auc:0.94268
[900]	validation_0-auc:0.942507
Stopping. Best iteration

In [22]:
sub['isFraud'] = preds
sub.to_csv("submission_xgb.csv", index=False)

## CatBoost

In [23]:
%%time

oof = np.zeros(len(X))
preds = np.zeros(len(X_test))

skf = GroupKFold(n_splits=6)
for i, (idxT, idxV) in enumerate(skf.split(X, y, groups=X['DT_M'])):
    month = X.iloc[idxV]['DT_M'].iloc[0]
    print('Fold',i,'withholding month',month)
    print('rows of train =',len(idxT),'rows of holdout =',len(idxV))
    
    clf = cb.CatBoostClassifier(iterations=500, learning_rate = 0.05, max_depth=12, class_weights=[1,2.5], 
                                cat_features=cat_features, objective='Logloss', eval_metric = 'AUC', task_type = 'GPU')

    h = clf.fit(X[cols].iloc[idxT], y.iloc[idxT], 
            eval_set=[(X[cols].iloc[idxV],y.iloc[idxV])],
            verbose=100, early_stopping_rounds=200)

    oof[idxV] += clf.predict_proba(X[cols].iloc[idxV])[:,1]
    preds += clf.predict_proba(X_test[cols])[:,1]/skf.n_splits
    
    del h, clf
    x = gc.collect()

print('#'*20)
print ('XGB95 OOF CV=',roc_auc_score(y,oof))

Fold 0 withholding month 12
rows of train = 453219 rows of holdout = 137321
0:	learn: 0.8502220	test: 0.8249917	best: 0.8249917 (0)	total: 562ms	remaining: 4m 40s
100:	learn: 0.9497446	test: 0.8879726	best: 0.8879726 (100)	total: 53.8s	remaining: 3m 32s
200:	learn: 0.9692325	test: 0.9011456	best: 0.9011665 (199)	total: 1m 46s	remaining: 2m 38s
300:	learn: 0.9800316	test: 0.9064892	best: 0.9065185 (299)	total: 2m 40s	remaining: 1m 46s
400:	learn: 0.9869171	test: 0.9095567	best: 0.9095989 (399)	total: 3m 34s	remaining: 53s
499:	learn: 0.9914863	test: 0.9102088	best: 0.9105900 (490)	total: 4m 28s	remaining: 0us
bestTest = 0.910589993
bestIteration = 490
Shrink model to first 491 iterations.
Fold 1 withholding month 15
rows of train = 488908 rows of holdout = 101632
0:	learn: 0.8458393	test: 0.8267531	best: 0.8267531 (0)	total: 525ms	remaining: 4m 21s
100:	learn: 0.9467381	test: 0.9074197	best: 0.9074197 (100)	total: 54.8s	remaining: 3m 36s
200:	learn: 0.9659796	test: 0.9181885	best: 0.918

In [24]:
sub['isFraud'] = preds
sub.to_csv("submission_cat.csv", index=False)

## LGBM

Will blend with a submission file later